In [2]:
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn

In [3]:
class NN(nn.Module):
    def __init__(self):
        super(NN, self).__init__()
        self.net = torch.nn.Sequential(
            nn.Linear(2, 20),
            nn.Tanh(),
            nn.Linear(20, 30),
            nn.Tanh(),
            nn.Linear(30, 30),
            nn.Tanh(),
            nn.Linear(30, 20),
            nn.Linear(20, 30),
            nn.Tanh(),
            nn.Linear(20, 1)
        )
    
    def forward(self, x):
        out = self.net(x)
        return out

In [ ]:
model = NN

In [ ]:
class Net:
    def __init__(self):
        device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        
        self.h = 0.1
        self.k = 0.1

        x = torch.arange(-1, 1 + self.h, self.h)
        t = torch.arange(-1, 1 + self.k, self.k)

        self.X = torch.stack(torch.meshgrid(x, t)).reshape(2, -1).T

        # Train data (boundary conditions)
        bc1 = torch.stack(torch.meshgrid(x[0], t)).reshape(2, -1).T
        bc2 = torch.stack(torch.meshgrid(x[-1], t)).reshape(2, -1).T

        ic = torch.stack(torch.meshgrid(x, t[0])).reshape(2, -1).T

        self.X_train = torch.concat([bc1, bc2, ic])

        y_bc1 = torch.zeros(len(bc1))
        y_bc2 = torch.zeros(len(bc2))
        
        y_ic = -torch.sin(math.pi * ic[:, 0])

        self.y_train = torch.cat([y_bc1, y_bc2, y_ic])
        self.y_train = self.y_train.unsqueeze(1)

        self.X = self.X.to(device=device)
        self.y_train = self.y_train.to(device=device)
        self.X_train = self.X_train.to(device=device)
        self.X.requires_grad = True

        # Optimizer
        self.optimizer_1 = torch.optim.Adam(self.model.parameters())
        self.optimizer_2 = torch.optim.lbfgs(self.model.parameters(), lr=1.0, max_iter=50000, max_ev=50000, history_size=50, tolerance_grad=1e-7, tolerance_change=1.0 * np.finfo(float).eps, line_search_fn="strong_wolfe")
